In [1]:
print('hola')

hola


In [5]:
import yfinance as yf
import pandas as pd
import numpy as np

djia_tickers = [
    "AAPL", "AMGN", "AXP", "BA", "CAT", "CRM", "CSCO", "CVX", "DIS", 
    "GS", "HD", "HON", "IBM", "INTC", "JNJ", "JPM", "KO", "MCD", "MMM",
    "MRK", "MSFT", "NKE", "PG", "TRV", "UNH", "V", "VZ", "WMT", "AMZN"
]

print("Descargando precios...")
datos = yf.download(djia_tickers, period="15y", interval="1d")

# Extracción robusta de los precios de cierre
try:
    # Intenta buscar en el primer nivel (comportamiento clásico)
    precios = datos['Adj Close']
except KeyError:
    try:
        # Intenta buscar en el segundo nivel usando cross-section (comportamiento nuevo)
        precios = datos.xs('Adj Close', level=1, axis=1)
    except KeyError:
        # Fallback por si la métrica fue renombrada simplemente a 'Close'
        try:
            precios = datos['Close']
        except KeyError:
            precios = datos.xs('Close', level=1, axis=1)

# 1. Eliminamos acciones que no existían hace 15 años (ej. si tienen más de 10 NaNs)
precios = precios.dropna(axis=1, thresh=len(precios) - 10)

# 2. Calculamos retornos logarítmicos
retornos_log = np.log(precios / precios.shift(1))

# 3. Borramos días aislados con fallos
retornos_log = retornos_log.dropna(axis=0, how='any')

print(f"Dimensiones finales: {retornos_log.shape[0]} días, {retornos_log.shape[1]} acciones.")
retornos_log.to_csv("retornos_log_djia_15y.csv")
print("Archivo 'retornos_log_djia_15y.csv' generado con éxito.")

Descargando precios...


[*********************100%***********************]  29 of 29 completed

Dimensiones finales: 3771 días, 29 acciones.
Archivo 'retornos_log_djia_15y.csv' generado con éxito.
